# 01 · Python 5 件套

> **学习目标**：在 45 分钟内，把后续做 RAG / Agent / 训练时**几乎天天用**的 5 个 Python 特性手撸一遍 —— list 推导、生成器、装饰器、`with` 上下文、`asyncio.gather`。
>
> **预备**：会写 Python 基础语法、知道函数是什么。
>
> **为什么重要**：写 LLM 代码时，这 5 个特性出现频率约 80%。看不懂别人的 RAG/Agent 代码，多半是这 5 个不熟。

## 1. List 推导式 — 比 for 短，比 map 快

心智模型：`[<表达式> for <元素> in <集合> if <条件>]`

**何时用**：想从一个集合做映射 / 过滤 / 组合，得到新集合。
**何时不用**：逻辑超过 1 行 → 退回普通 for（可读性优先）。

In [ ]:
# 反例：啰嗦的 for 循环
squares = []
for x in range(10):
    if x % 2 == 0:
        squares.append(x * x)
print('for-loop  ', squares)

# 正例：list 推导
squares2 = [x * x for x in range(10) if x % 2 == 0]
print('list-comp ', squares2)

assert squares == squares2

In [ ]:
# 嵌套：把 2D 矩阵 flatten
matrix = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
flat = [v for row in matrix for v in row]
print(flat)

# 同时来个 dict 推导，做 token -> id 映射
tokens = ['<bos>', 'hello', 'world', '<eos>']
tok2id = {t: i for i, t in enumerate(tokens)}
print(tok2id)

## 2. 生成器 — 用时间换空间

**核心区别**：list 一次性把所有元素塞进内存；生成器只在你 `next()` / `for` 迭代时**算一次给一个**。

**LLM 场景**：流式读 1000 万行日志、流式接收 LLM token、训练时按需加载样本 —— 全靠生成器。

In [ ]:
import sys

# list：100 万元素全装内存
big_list = [i * i for i in range(1_000_000)]
print('list 内存约', sys.getsizeof(big_list) / 1024 / 1024, 'MB')

# 生成器表达式：括号换圆，立刻省 99.99%
big_gen = (i * i for i in range(1_000_000))
print('gen  内存约', sys.getsizeof(big_gen), 'bytes')

# 但 gen 只能迭代一次！
print('first sum:', sum(big_gen))
print('second sum:', sum(big_gen), '<- 已经被耗尽')

In [ ]:
# 自定义生成器：yield 关键字
def stream_chunks(text: str, size: int):
    """模拟 RAG 里的文本切块。yield 一段就出一段，不全部算完。"""
    for i in range(0, len(text), size):
        yield text[i:i + size]

doc = '人工智能时代的检索增强生成是当下最热门的方向之一'
for idx, chunk in enumerate(stream_chunks(doc, 6)):
    print(f'chunk {idx}: {chunk!r}')

## 3. 装饰器 — 在函数前后插入逻辑

**心智模型**：`@deco` 等价于 `f = deco(f)`。装饰器**把函数当参数收进来、返回一个新函数**。

**LLM 场景**：给所有 LLM 调用自动加重试 / 限流 / 缓存 / trace —— 装饰器最爽。

In [6]:
import time
from functools import wraps

def timeit(fn):
    """通用计时装饰器。@wraps 保留原函数 __name__/__doc__。"""
    @wraps(fn)
    def wrapped(*args, **kwargs):
        t0 = time.perf_counter()
        result = fn(*args, **kwargs)
        dt = (time.perf_counter() - t0) * 1000
        print(f'[{fn.__name__}] {dt:.2f} ms')
        return result
    return wrapped

@timeit
def fake_llm_call(prompt: str) -> str:
    time.sleep(0.1)
    return f'answer to: {prompt}'

print(fake_llm_call('什么是 RAG？'))

[fake_llm_call] 100.73 ms
answer to: 什么是 RAG？


In [ ]:
# 带参装饰器：再外套一层
def retry(times: int = 3):
    def deco(fn):
        @wraps(fn)
        def wrapped(*args, **kwargs):
            last_err = None
            for i in range(times):
                try:
                    return fn(*args, **kwargs)
                except Exception as e:
                    last_err = e
                    print(f'  retry {i+1}/{times} after error: {e}')
            raise last_err
        return wrapped
    return deco

import random

@retry(times=4)
def flaky_api():
    if random.random() < 0.7:
        raise RuntimeError('网络抖动')
    return 'ok'

random.seed(1)
print('result:', flaky_api())

## 4. `with` 上下文管理器 — 资源生命周期

**核心问题**：打开了的东西必须关、申请了的锁必须释放、占用的 GPU 显存必须还。`with` 保证哪怕中途抛异常，**清理代码也会被执行**。

**LLM 场景**：torch 的 `with torch.no_grad():`、tokenizer 的临时 padding、训练时的 `with torch.cuda.amp.autocast():` 全是这套机制。

In [ ]:
# 反例：忘了关 —— 用一个临时文件演示，避免对 cwd 的假设
from pathlib import Path
tmp = Path('./_demo.txt')
tmp.write_text('Hello, with-context!\n', encoding='utf-8')

f = open(tmp, 'r', encoding='utf-8')
# ... 一旦下面这行抛异常，f 就泄漏了
_ = f.read()
f.close()

# 正例：with 保证 close
with open(tmp, 'r', encoding='utf-8') as f:
    data = f.read()
    print('读到', len(data), '字节:', data.strip())
# 退出 with 块时自动 close —— 哪怕中间抛异常

tmp.unlink()

In [ ]:
# 自定义 context manager：用 contextlib 最简洁
from contextlib import contextmanager

@contextmanager
def llm_session(name: str):
    print(f'>>> 开 session {name}')
    try:
        yield {'name': name, 'tokens': 0}   # yield 的就是 with ... as x 里的 x
    finally:
        print(f'<<< 关 session {name}（保证执行）')

with llm_session('test') as sess:
    sess['tokens'] += 42
    print('  内部使用 sess =', sess)
    # 故意抛异常，看 finally 是否执行
    # raise ValueError('炸了')

## 5. `asyncio.gather` — 并发 LLM 调用的基石

**核心理解**：CPU 等网络 / 等磁盘的时候是闲着的；`async/await` 让你在等的间隙**切去做别的任务**，把这些等待时间叠起来用。

**LLM 场景**：同时调 10 个 LLM、并发查 5 个向量库、批量爬 100 个网页 —— 不用 async 直接挂在最慢的那个上。

In [4]:
import asyncio, time

async def fake_llm(prompt: str, delay: float = 0.5) -> str:
    # 用 asyncio.sleep 模拟「网络等待」—— 在等的时候 event loop 可以跑别的协程
    await asyncio.sleep(delay)
    return f'answer({prompt})'

async def main():
    prompts = ['Q1', 'Q2', 'Q3', 'Q4', 'Q5']

    # 串行：5 * 0.5s = 2.5s
    t0 = time.perf_counter()
    serial = [await fake_llm(p) for p in prompts]
    print(f'串行: {time.perf_counter() - t0:.2f}s', serial)

    # 并发：max(0.5s) ≈ 0.5s
    t0 = time.perf_counter()
    parallel = await asyncio.gather(*[fake_llm(p) for p in prompts])
    print(f'并发: {time.perf_counter() - t0:.2f}s', parallel)

await main()    # Jupyter 里可直接 await；普通 .py 里要 asyncio.run(main())

串行: 2.57s ['answer(Q1)', 'answer(Q2)', 'answer(Q3)', 'answer(Q4)', 'answer(Q5)']
并发: 0.50s ['answer(Q1)', 'answer(Q2)', 'answer(Q3)', 'answer(Q4)', 'answer(Q5)']


In [ ]:
# 真实场景：用 Semaphore 限并发（防止把 LLM provider 打挂）
async def limited_llm(prompt, sem, delay=0.3):
    async with sem:                # async-friendly with，最大并发由 sem 控制
        await asyncio.sleep(delay)
        return prompt.upper()

async def batch():
    num_task = 10
    num_task_run = 6
    sem = asyncio.Semaphore(num_task_run)     # 同时最多 num task run 个在跑
    tasks = [limited_llm(f'q{i}', sem) for i in range(num_task)]
    t0 = time.perf_counter()
    results = await asyncio.gather(*tasks)
    print(f'{num_task}个任务，限并发 {num_task_run}：{time.perf_counter() - t0:.2f}s')
    print(results)

await batch()

10 个任务，限并发 3：0.62s
['Q0', 'Q1', 'Q2', 'Q3', 'Q4', 'Q5', 'Q6', 'Q7', 'Q8', 'Q9']


## 深入思考

1. **list 推导 vs 生成器**：什么时候选哪个？
   - 答：结果要复用 / 要随机访问 → list；只迭代一次 / 内存敏感 → 生成器。
2. **装饰器叠加顺序**：`@a @b def f()` 的执行顺序是先 `a` 还是先 `b`？
   - 答：等价于 `f = a(b(f))`，所以**调用时先进 `a`，再到 `b`，最后是真函数**。试试加 print 看顺序。
3. **`asyncio.gather` 里某个任务抛异常会怎样？**
   - 默认：整个 gather 立刻 raise，其它任务被 cancel。想容错？传 `return_exceptions=True`。
4. **`with` 和 `try/finally` 等价吗？**
   - 接近。但 `__exit__` 还能根据是否有异常做条件清理，比 finally 更灵活。

试着改最后一个 cell：把 `Semaphore(3)` 改成 `Semaphore(1)`，看耗时是不是变成 10 × 0.3s。

## 自检 ✅

- [ ] 不看代码，10 秒内写出「把 list 里所有偶数翻倍」的 list 推导。
- [ ] 解释「为什么生成器只能迭代一次」。
- [ ] 不查文档写一个 `@cache_result(ttl=60)` 装饰器骨架（不用真存）。
- [ ] 解释 `with` 比 `try/finally` 优势在哪。
- [ ] 解释为什么 `time.sleep()` 在 `async` 函数里**没用**（必须 `asyncio.sleep`）。

## 下一步

→ [`02_numpy_basics.ipynb`](02_numpy_basics.ipynb)